In [ ]:
%pip install llama-index llama-index-embeddings-huggingface llama-index-llms-huggingface llama-index-llms-groq pytesseract pdf2image transformers bitsandbytes accelerate
!pip -q install pythainlp
!pip install attacut

In [ ]:
import os

from llama_index.core import PromptTemplate, Document, SimpleDirectoryReader
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import huggingface_hub

from pdf2image import convert_from_path

from tqdm.notebook import tqdm

import torch

from typing import List, Any, Optional
import re
from pythainlp import word_tokenize
from pythainlp.util import normalize
from pythainlp.corpus import thai_stopwords

from llama_index.core import Document, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
from pathlib import Path
from llama_index.core import VectorStoreIndex, Document
from llama_index.core import StorageContext, load_index_from_storage

DATA_DIR = Path("/content/pdfs")
INDEX_DIR = Path("/content/rag_index")
INDEX_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
documents = SimpleDirectoryReader(DATA_DIR).load_data()

In [ ]:
from collections import Counter

def get_fname(d):
    md = getattr(d, "metadata", {}) or {}
    return md.get("file_name") or md.get("filename") or md.get("source")

files = [get_fname(d) for d in documents]
print(Counter(files))  # นับจำนวน document ต่อไฟล์

In [ ]:
# ---------- Clean & Tokenize ----------
def _clean_text(text: str,
                lowercase: bool = True,
                keep_punct: bool = True,
                keep_numbers: bool = True) -> str:
    if not text:
        return ""
    t = normalize(text)
    if lowercase:
        t = t.lower()
    if not keep_punct:
        t = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9\s]", " ", t)
    if not keep_numbers:
        t = re.sub(r"\b\d+\b", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def _tokenize_th(text: str,
                 engine: str = "attacut",
                 remove_stopwords: bool = False,
                 extra_stopwords: Optional[set] = None,
                 min_token_len: int = 1) -> str:
    """ตัดคำแล้ว join ด้วยช่องว่าง (เหมาะกับการฝังเวคเตอร์)"""
    toks = word_tokenize(text, engine=engine, keep_whitespace=False)
    if remove_stopwords:
        sw = thai_stopwords()
        if extra_stopwords:
            sw = sw.union(extra_stopwords)
        toks = [t for t in toks if t not in sw]
    toks = [t.strip() for t in toks if t and len(t) >= min_token_len]
    return " ".join(toks)

# ---------- Main: documents -> nodes ready for VectorStoreIndex ----------
def prepare_nodes_for_vector_index(
    documents: List[Any],
    *,
    engine: str = "attacut",
    remove_stopwords: bool = False,
    extra_stopwords: Optional[set] = None,
    lowercase: bool = False,
    keep_punct: bool = True,
    keep_numbers: bool = True,
    min_token_len: int = 1,
    chunk_size: int = 900,
    chunk_overlap: int = 120
) -> List[TextNode]:
    """รับ LlamaIndex documents (โหลดจาก SimpleDirectoryReader) และคืน nodes ที่พร้อมสร้าง VectorStoreIndex"""
    # 1) แปลงเอกสารเป็น Document ใหม่ที่ผ่านการ clean+tokenize แล้ว (เก็บ metadata เดิม)
    processed_docs: List[Document] = []
    for d in documents:
        raw = getattr(d, "text", "") or ""
        meta = dict(getattr(d, "metadata", {}) or {})
        cleaned = _clean_text(
            raw,
            lowercase=lowercase,
            keep_punct=keep_punct,
            keep_numbers=keep_numbers,
        )
        tokenized = _tokenize_th(
            cleaned,
            engine=engine,
            remove_stopwords=remove_stopwords,
            extra_stopwords=extra_stopwords,
            min_token_len=min_token_len
        )
        processed_docs.append(Document(text=tokenized, metadata=meta))

    # 2) Chunk เป็น nodes (เหมาะกับการทำดัชนี/ค้นคืน)
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    nodes: List[TextNode] = splitter.get_nodes_from_documents(processed_docs)
    return nodes

# ---------- Usage ----------
# สมมติคุณมีตัวแปร 'documents' จาก SimpleDirectoryReader แล้ว:
# documents = SimpleDirectoryReader("/content/pdfs").load_data()

# 1) เตรียม nodes พร้อมเข้าดัชนี
# (ปรับพารามิเตอร์ได้ตามต้องการ เช่น engine="attacut" ถ้าติดตั้งไว้)
nodes = prepare_nodes_for_vector_index(
    documents,
    engine="attacut",
    remove_stopwords=True,
    lowercase=True,
    keep_punct=True,
    keep_numbers=True,
    min_token_len=1,
    chunk_size=700,
    chunk_overlap=283,
)

In [ ]:
from llama_index.llms.groq import Groq
from google.colab import userdata

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
Settings.embed_model = embed_model

In [ ]:
index = VectorStoreIndex(nodes)

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-v2-m3", top_n=5
)

In [ ]:
llm = Groq(model="meta-llama/llama-4-maverick-17b-128e-instruct", api_key= userdata.get("Groq"), temperature=0.1, num_beams=8, max_length=2048)

Settings.llm = llm

In [ ]:
class DummyQueryBundle:
    def __init__(self, query_str):
        self.query_str = query_str

In [ ]:
from llama_index.core.indices.vector_store.base import VectorStoreIndex
from llama_index.core import PromptTemplate
from llama_index.core.response_synthesizers import TreeSummarize

from llama_index.core.workflow import Workflow, step, StartEvent, StopEvent
import re

text2text_template = PromptTemplate(
"""Your are expert linguistics.
You will extract the important part of the input
Just answer one Answer:

EX.
Input: อยากสอบถามข้อมูลหน่อยครับ วิชาเฉพาะด้านของวิทยาการคอมพิวเตอร์มีวิชาอะไรบ้างเหรอครับ
Output: วิชาเฉพาะด้านของวิทยาการคอมพิวเตอร์มีวิชาอะไรบ้าง

Input: {query_str}
Answer:
"""

)

refine_template = PromptTemplate(
    """
    -สรุปข้อความเป็นภาษาไทยให้อ่านง่าย และไม่สั้นหรือไม่ยาวเกิน
    -ถ้าข้อความเป็นชื่อวิชาให้ระบุ ชื่อวิชาโดยใช้ภาษาอังกฤษเท่านั้น พร้อมรหัสวิชา
    -ถ้าข้อความสามารถแยกเป็นข้อๆได้ ให้แยกเป็นข้อๆ
    -ถ้ามีวิชาไหนอยู่ในคำตอบให้ใส่มาให้ครบทุกวิชา
    -ถ้าส่วนไหนของคำตอบไม่เกี่ยวข้องกับในคำถามให้ตัดออก

    Ins: {query_str}
    Input: {quried_text}
    Ans:
    """
)

def run_query_pipeline(query_str: str):
    # 1. Use text2text prompt to transform the query (optional)
    # (You may or may not need this step depending on your use-case)
    prompt = text2text_template.format(query_str=query_str)
    # send to llm
    intermediate = llm.complete(prompt)
    query_text = intermediate.text  # ✅ Extract the string

    retriever = index.as_retriever(similarity_top_k=5)
    nodes = retriever.retrieve(query_text)

    # 3. rerank
    query_bundle = DummyQueryBundle(query_text)
    reranked = reranker.postprocess_nodes(nodes, query_bundle=query_bundle)
    # 4. Summarize / synthesize

    summary = TreeSummarize()

    summary_response = summary.synthesize(query=intermediate.text.strip(), nodes=reranked)
    # 5. Finally refine the summary (if needed)
    refine_prompt = refine_template.format(query_str=summary_response)
    final = llm.complete(refine_prompt)
    return final

In [ ]:
# Updated Gradio Interface
import gradio as gr

import gc # Import gc for garbage collection
import shutil # Import shutil
import torch # Import torch

with gr.Blocks(title="RAG over PDFs") as demo:
    gr.Markdown("## 📚 RAG over PDFs")
    with gr.Tab("Chat"):
        gr.Markdown("Ask questions grounded in your PDFs.")

        question = gr.Textbox(label="Your question")
        answer = gr.Textbox(label="Answer", interactive=False, lines=10)

        def on_ask(q, evt: gr.EventData): # Add r for rerank_top_n and evt for the event object
            try:
                return run_query_pipeline(query_str= q)
            except Exception as e:
                import traceback
                traceback.print_exc() # Print traceback for debugging
                return f"❌ Error during RAG process: {e}", []

        ask_btn = gr.Button("🤖 Ask")
        ask_btn.click(
            on_ask,
            inputs=[question],
            outputs=[answer]
        )


demo.queue().launch(debug=True, share=True)